# ORBIT — Phase 3A: Real-Game Gameplay Collection & Verification

**Objective (Step 1):** Let the Gate 6 H=4 policy play 30 real games of Crafter with controlled exploration ($\tau=1.2$). Record all visual transitions, verify schema boundaries, save to disk, and confirm reloadability without updating any model weights.

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or Update Repository & Install Dependencies

In [ ]:
!git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world 2>/dev/null || (cd /content/learn-a-world && git pull origin main)
%cd /content/learn-a-world

!pip install -q -r requirements.txt
!pip install -q torchvision timm matplotlib

## 3. Verify Compute Environment

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('Running on CPU. For faster rollouts, set Runtime -> Change runtime type -> T4 GPU.')

## 4. Run Phase 3A: Collect 30 Real Episodes & Verify Disk Serialization

In [ ]:
%cd /content/learn-a-world

# Ensure output directories exist
!mkdir -p data /content/drive/MyDrive/learn-a-world/data

# Run Phase 3A standalone collection and integrity verification
!python scripts/run_phase3a_collection.py \
  --policy-checkpoint results/gate6/policy_h4.pt \
  --episodes 30 \
  --start-seed 1000 \
  --max-steps 300 \
  --temperature 1.2 \
  --device auto \
  --output data/phase3a_selfplay_30eps.npz

# Copy verified replay to Google Drive
!cp data/phase3a_selfplay_30eps.npz /content/drive/MyDrive/learn-a-world/data/
!cp data/phase3a_verification_report.json /content/drive/MyDrive/learn-a-world/data/
print('Successfully backed up Phase 3A replay data to Google Drive.')

## 5. Display Phase 3A Verification Report

In [ ]:
import json

with open('data/phase3a_verification_report.json') as f:
    rep = json.load(f)

print('=' * 80)
print(f"PHASE 3A ACCEPTANCE REPORT: {rep['status']}")
print('=' * 80)
print(f"Episodes Collected:        {rep['num_episodes']}")
print(f"Total Frames:              {rep['total_frames']}")
print(f"Mean Return:               {rep['mean_return']:.2f} ± {rep['std_return']:.2f}")
print(f"Mean Steps / Episode:      {rep['mean_steps']:.1f}")
print(f"Movement Percentage:       {rep['movement_percentage']:.1f}%")
print(f"Action Shannon Entropy:    {rep['action_entropy']:.3f}")
print(f"File Size:                 {rep['file_size_mb']:.2f} MB")
print('-' * 80)
print('Acceptance Criteria Checklist:')
for item, val in rep['checklist'].items():
    print(f"  [{'PASS' if val else 'FAIL'}] {item.replace('_', ' ').capitalize()}")
print('=' * 80)